In [1]:
!python -m pip install --user datasets


[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from datasets import load_dataset


C:\Users\mansh\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# !pip install -U g4f
# !pip install aiohttp
!python -m pip install --user -U g4f
!python -m pip install --user aiohttp


[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import time
from g4f.client import Client

# Initialize the client
client = Client()

# List of input messages
messages = [
    "Tell me a joke on ML engineer"
]

total_time = 0
retry_limit = 3  # Maximum retries for a failed request

# Loop through each message, send it, and measure response time
for i, msg in enumerate(messages):
    print(f"Processing message {i + 1} of {len(messages)}...")

    for attempt in range(retry_limit):
        try:
            start_time = time.time()  # Start the timer

            # API request
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": msg}],
                web_search=False
            )

            end_time = time.time()  # End the timer
            elapsed_time = end_time - start_time
            total_time += elapsed_time

            print(f"Message {i + 1}: {msg}")
            print(f"Response: {response.choices[0].message.content}")
            print(f"Time taken: {elapsed_time:.2f} seconds\n")
            break  # Exit retry loop on success

        except Exception as e:
            print(f"Attempt {attempt + 1} failed with error: {e}")
            if attempt + 1 == retry_limit:
                print(f"Skipping message {i + 1} after {retry_limit} attempts.\n")
            else:
                print("Retrying...\n")
                time.sleep(1)  # Add delay before retrying

# Summary
print(f"Total time for {len(messages)} messages: {total_time:.2f} seconds")
if messages:
    print(f"Average time per message: {total_time / len(messages):.2f} seconds")


Processing message 1 of 1...
Message 1: Tell me a joke on ML engineer
Response: Why did the machine learning engineer break up with their partner?

Because they had too many "overfitting" issues!
Time taken: 2.00 seconds

Total time for 1 messages: 2.00 seconds
Average time per message: 2.00 seconds


In [5]:
# for 5.65 model
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor
from g4f.client import Client

# Synchronous function to send a request
def send_request_sync(client, message):
    """
    Sends a synchronous request to the GPT API.
    """
    while True:  # Retry until the request succeeds
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": message}],
                web_search=False
            )
            generated_response = response.choices[0].message.content
            print(f"Generated Response: {generated_response}")
            return generated_response
        except Exception as e:
            print(f"Error sending request: {e}")
            time.sleep(1)  # Wait for a short time before retrying

# Asynchronous function to execute the request
async def send_request(client, message, executor):
    """
    Asynchronous wrapper for the synchronous API request.
    """
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(executor, send_request_sync, client, message)

# Function to generate the prompt for grading
def create_prompt(s_no, partial_story, completed_story):
    """
    Generates the grading prompt based on the partial and completed stories.
    """
    return f"""The following exercise, the student is given a beginning of a text. The student needs to complete the partially completed text.
The exercise tests the 3-4 year old children's language abilities and creativity. 
Serial Number: {s_no}
Partial Story: {partial_story}
Completed Story: {completed_story}

Now, grade the children's completion in terms of grammar, creativity, consistency, meaningfulness, plot each out of 10 with the text's beginning and
completed text. INCLUDE THE SERIAL NUMBER OF THE STORY WHILE RESPONDING. Also tell the total of the scores. Moreover, please provide your best guess of what the age of the student might be,
as reflected from the completion. Choose from possible age groups: A: 3 or under. B: 4-5. C: 6-7. D: 8-9. E:
10-12. F: 13-16."""

# Main function to process a list of stories
async def main():
    # Define your list of tuples (partial_story, completed_story)
    stories = [
    ("1", "In a dark basement, there is a white toilet sitting on some steps. Around it, there are broken boards","<start>In a dark basement, there is a white toilet sitting on some steps. Around it, there are broken boards and a big book with a big smile. The sun is shining, and the sky is blue and shiny. The sun is shining, and the sun shines bright, making everything look special. It feels like a fun place where the water splashes are waiting for someone to have fun to see!<pad>"),
    ("2", "In a shiny bathroom, the walls sparkle like magic! There’s a big tree next to a toilet, and plants","<start>In a shiny bathroom, the walls sparkle like magic! There’s a big tree next to a toilet, and plants are so colorful and some colorful books. The cat looks so cool and cool as they are sitting on a soft bed. The sink is soft and soft fur and some cooked and soft fur. It feels like a fun place where you can see the can see the cat see the cat the cake. I wonder what stories the cat taste the cake and what is a fun place!<pad>"),
    ("3", "There is a big table full of yummy treats! There are cookies, cakes, and colorful candies all together.","<start>There is a big table full of yummy treats! There are cookies, cakes, and colorful candies all together. The bed is soft and soft fur are some colorful and some are shiny and some shiny signs. The sun is shining, and the sun shines bright and some special. It feels like a fun place where the water splashes are waiting for the beach. I wonder what the ball will go next!<pad>"),
    ("4", "Pink cakes and lollipops rest on white tables and pink plates. Cake slices have forks beside them.","<start>Pink cakes and lollipops rest on white tables and pink plates. Cake slices have forks beside them. The sun is shining, and the sky is blue and shiny. The sun is shining, and the sky is blue and shiny. The sun shines bright, and the sun shines bright, making everything look so much fun! I wonder what they are having a great time together!<pad>"),
    ("5", "The cake is so colorful with chocolate and white frosting. It has cute polka dots and sprinkles on","<start>The cake is so colorful with chocolate and white frosting. It has cute polka dots and sprinkles on the soft grass. The sun is shining, and the sky is blue and shiny. The sun is shining, and the sky is blue and shiny. The sun shines bright, and the sky is blue and seems happy. I wonder what the ball will go next! Maybe they are having a great time together!<pad>"),
    ("6", "In a funny bathroom, there are two shiny toilets side by side. The floor is like a big","<start>In a funny bathroom, there are two shiny toilets side by side. The floor is like a big blanket waiting for someone to go. The counter is soft and smells so good! The counter is soft and soft fur and some cookies are soft and smiles. The sun shines bright, making everything look special. I wonder what the cat is waiting for someone to see the cake and where they are going and want to eat!<pad>"),
    ("7", "In a happy green bathroom, there are funny monkeys on the curtain. A shiny sink and a little toilet","<start>In a happy green bathroom, there are funny monkeys on the curtain. A shiny sink and a little toilet that looks very cozy. The bear is so cool and soft fur and some cookies are so cool. The baby is soft and soft fur away, and they look so happy. The sun shines bright, making everything look special. I wonder what they are going to see the can see the sun shines bright and have fun!<pad>"),
    ("8", "The bathroom has a white toilet and a tall sink. There's also a shower, but the walls look like","<start>The bathroom has a white toilet and a tall sink. There's also a shower, but the walls look like a big bird. The sun is shining bright, and it looks so cool! The sun shines bright, and the sky is blue and shiny. The sun shines bright, and the sun is shining bright. It feels like a fun place where the water splashes are waiting for the ball to see the ball flying through the warm sun!<pad>"),
    ("9", "In a shiny bathroom, there is a big toilet bowl sitting on the floor. A tall stall has a","<start>In a shiny bathroom, there is a big toilet bowl sitting on the floor. A tall stall has a shiny white sink that looks very cozy. The cat is sitting on a soft bed, waiting for someone to go. The cat is sitting on the sidewalk, like a big blanket. The bathroom is so cool and special places to see the cake. It feels like a fun place where the water are waiting for someone to see the cake.<pad>"),
    ("10", "There's a man on a shiny, old motorcycle. He wears a long coat and sits very still. The picture","<start>There's a man on a shiny, old motorcycle. He wears a long coat and sits very still. The picture is so cool and sparkly in the sunlight. The sun is shining, and the sun is shining bright. The man is smiling and laughing as they walk on the water. It looks like he is having fun on the water! I wonder what he is thinking about while he is having fun on the water!<pad>"),
    ("11", "There's a big building with a clock inside that's tall and pointy. The church has a tall tower that","<start>There's a big building with a clock inside that's tall and pointy. The church has a tall tower that looks like a big bird. The sun is shining, and the sky is blue and shiny. The sun is shining, and the sky is blue and seems happy. It feels like a fun party with the ball with the ball flowers and shiny tracks. I wonder what the ball will go next!<pad>"),   
    ("12", "The green bowl is on the table. It is full of little trees called broccoli. Some broccoli is green","<start>The green bowl is on the table. It is full of little trees called broccoli. Some broccoli is green and some are shiny and shiny. The sun is shining, and the sun is shining bright. The sun shines bright, and the sun shines bright and smiles. It feels like a fun party with the ball flowers and something fun. I wonder what they are having a great time together!<pad>"),
    ("13", "There is a big, yummy cake on a shiny silver plate. It has white frosting and blue sprinkles in","<start>There is a big, yummy cake on a shiny silver plate. It has white frosting and blue sprinkles in the sun. The cat is sitting on a big book on the sidewalk, and it looks very cozy. The cat is sitting on the sidewalk, and it looks so cool! I wonder what the cat is waiting for someone to eat the cake. I wonder what the cat is waiting for someone to see the cake!<pad>"),
    ("14", "A big parade is happening! A police motorcycle zooms by with a shiny car behind it. People are watching","<start>A big parade is happening! A police motorcycle zooms by with a shiny car behind it. People are watching the ball with a big smile. The sun is shining, and the sun is shining bright. The sun is shining, and the sun is shining bright. The sun is shining, and the sun is shining bright. It feels like a fun adventure in the snow!<pad>"),
    ("15", "A fluffy cat is on a table. It leans over a shiny fish bowl, looking at the fish swimming","<start>A fluffy cat is on a table. It leans over a shiny fish bowl, looking at the fish swimming in the sun. The sun is shining, and the sky is blue and white, and it looks so cool! The sun is shining, and the sky is blue. I wonder what the will go next to the snow! It looks like they are having a great time together!<pad>"),
    ("16", "The orange kitty sits on the table beside a bright bowl. It lays on the table, looking cozy and","<start>The orange kitty sits on the table beside a bright bowl. It lays on the table, looking cozy and happy. The baby is smiling and laughing as the ball to the train the ball. The sun is shining, and the sun shines bright and smiles. It looks like they are having a great time together. I wonder what the ball will go next!<pad>"),
    ("17", "The kitty is very funny. It stands in an empty bowl while munching from another one. The dishes are","<start>The kitty is very funny. It stands in an empty bowl while munching from another one. The dishes are shiny and shiny, and it looks very curious. The sun is shining, and it looks so cool! The sun is shining, and the sun is shining bright. I wonder what the cat is waiting for someone to see the cake. It feels like a fun place where the water splashes are waiting for someone to have fun time!<pad>"),
    ("18", "The cat is eating its food. It's funny because it's standing inside a bowl while munching. The cat","<start>The cat is eating its food. It's funny because it's standing inside a bowl while munching. The cat is sitting on a big board with a big smile. The sun is shining, and the sky is blue and shiny. The sun is shining, and the sun is shining bright. It looks like they are having a great time together! I wonder what the ball will go next!<pad>"),
    ("19", "A young man is sitting in a small room with a computer. He wears a cozy sweatshirt and looks", "<start>A young man is sitting in a small room with a computer. He wears a cozy sweatshirt and looks very cool. The man is smiling and laughing as they walk on the soft grass. The man is smiling and smiling as he walks the ball with his skateboard. It looks like he is having fun adventures with the ball flying on the water. I wonder what he is thinking about where they are having a great time!<pad>"),
    ("20", "The toilet has a big, round light above it. Sunlight comes in from a round window in the wall.","<start>The toilet has a big, round light above it. Sunlight comes in from a round window in the wall. The sun is shining, and the sky is blue and shiny. The sun is shining, and the sun is shining bright. The man is smiling and laughing as they walk on the water. It feels like a fun day for the water and splashes all around him. I wonder what the ball will go next!<pad>"),
    ("21", "In a tiny bathroom, there is a white toilet. The roof is slanted, and a bright window lets in","<start>In a tiny bathroom, there is a white toilet. The roof is slanted, and a bright window lets in the sun. The cat looks so cool and special places to sit and colorful. The baby is soft and soft food and smiles as they are having fun. The sink is so cool and special places to see the cake with a big smile. It feels like a fun place where you can see the can see the cat together!<pad>"),
    ("22", "There are tiny green beads and nuts inside bamboo pieces. I see scissors next to brussel sprouts and shiny","<start>There are tiny green beads and nuts inside bamboo pieces. I see scissors next to brussel sprouts and shiny street. The sun is shining, and the sun shines bright and shiny. The sun is shining, and the sun shines bright and some shiny signs. It feels like a fun party with the ball flowers and the ball flowers. I wonder what they are going to see!<pad>"),
    ("23", "A man sits at his desk with a big, yummy hotdog. He's wearing a blue shirt and working on","<start>A man sits at his desk with a big, yummy hotdog. He's wearing a blue shirt and working on his skateboard. The man is smiling and laughing as he walks that look like a big bird. The man is smiling and laughing as he walks the waves. It looks like he is having fun adventures with his skateboard! I wonder what he is thinking about while he is having fun on the water!<pad>"),
    ("24", "The bowl has yummy fruit like apples, bananas, and oranges. It's sitting on the wooden table. The blue plate","<start>The bowl has yummy fruit like apples, bananas, and oranges. It's sitting on the wooden table. The blue plate is so cool and some are shiny and smiles. The sun is shining, and the sky is blue and shiny. The sun is shining, and the sun is shining bright. It feels like a fun place where the water splashes are watching the ball. I wonder what the will go next!<pad>"),
    ("25", "In a big parking lot, two cool motorbikes sit next to a shiny car. One bike is blue with","<start>In a big parking lot, two cool motorbikes sit next to a shiny car. One bike is blue with a big smile on the soft grass, and they look very happy. The sun shines bright, and the sky is blue and shiny. The sun shines bright, and the sky is blue and shiny. It feels like a fun place where the water splashes are watching the waves. I wonder what they are having a great time!<pad>"),

]
    # Create an instance of the Client
    client = Client()

    # Create a ThreadPoolExecutor to run the blocking call asynchronously
    executor = ThreadPoolExecutor(max_workers=20)  # Adjust max_workers as needed for parallelism

    # Loop through the list of stories and send requests
    tasks = []
    for s_no, partial, completed in stories:
        prompt = create_prompt(s_no, partial, completed)
        task = send_request(client, prompt, executor)
        tasks.append((s_no, partial, completed, task))

    # Gather all results
    results = await asyncio.gather(*(task[3] for task in tasks))

    print("@" *100)
    print("\n printing the responses")

    data = {
        "Partial": [story[1] for story in tasks],
        "Complete": [story[2] for story in tasks]
    }
    df = pd.DataFrame(data)

    # Print the DataFrame
    print("Generated DataFrame:")
    print(df)
    df.to_csv("LongDesc_5.65M_short_partial_results.csv")

    # Print the final responses with serial numbers and stories
    for (s_no, partial, completed), response in zip(stories, results):
        print(f"Serial Number: {s_no}")
        print(f"Partial Story: {partial}")
        print(f"Completed Story: {completed}")
        print(f"Response: {response}")
        print("-" * 50)

# Run the main function
await main()


Generated Response: ### Story Evaluation

**Serial Number:** 13

#### Grades:

1. **Grammar:** 5/10
   - The completion contains several grammatical errors that disrupt the flow of the text. For example, the sentence "I wonder what the cat is waiting for someone to eat the cake" is grammatically incorrect.

2. **Creativity:** 6/10
   - The child exhibits some creativity by introducing a cat and placing it on the sidewalk with aspects such as a big book, but the completion lacks a more developed plot and depth in creativity.

3. **Consistency:** 5/10
   - The completion includes inconsistent elements, such as repetitive sentences about the cat and the sidewalk, and doesn't logically follow one cohesive thread from beginning to end.

4. **Meaningfulness:** 4/10
   - The completion has moments that don't make complete sense. The purpose of the cat and its relation to the cake are unclear, and there is repetition without adding to the story’s meaning.

5. **Plot:** 3/10
   - The plot devel

In [6]:
# for 16M model
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor
from g4f.client import Client

# Synchronous function to send a request
def send_request_sync(client, message):
    """
    Sends a synchronous request to the GPT API.
    """
    while True:  # Retry until the request succeeds
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": message}],
                web_search=False
            )
            generated_response = response.choices[0].message.content
            print(f"Generated Response: {generated_response}")
            return generated_response
        except Exception as e:
            print(f"Error sending request: {e}")
            time.sleep(1)  # Wait for a short time before retrying

# Asynchronous function to execute the request
async def send_request(client, message, executor):
    """
    Asynchronous wrapper for the synchronous API request.
    """
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(executor, send_request_sync, client, message)

# Function to generate the prompt for grading
def create_prompt(s_no, partial_story, completed_story):
    """
    Generates the grading prompt based on the partial and completed stories.
    """
    return f"""The following exercise, the student is given a beginning of a text. The student needs to complete the partially completed text.
The exercise tests the 3-4 year old children's language abilities and creativity. 
Serial Number: {s_no}
Partial Story: {partial_story}
Completed Story: {completed_story}

Now, grade the children's completion in terms of grammar, creativity, consistency, meaningfulness, plot each out of 10 with the text's beginning and
completed text. INCLUDE THE SERIAL NUMBER OF THE STORY WHILE RESPONDING. Also tell the total of the scores. Moreover, please provide your best guess of what the age of the student might be,
as reflected from the completion. Choose from possible age groups: A: 3 or under. B: 4-5. C: 6-7. D: 8-9. E:
10-12. F: 13-16."""

# Main function to process a list of stories
async def main():
    # Define your list of tuples (partial_story, completed_story)
    stories = [
    ("1", "In a dark basement, there is a white toilet sitting on some steps. Around it, there are broken boards","<start>In a dark basement, there is a white toilet sitting on some steps. Around it, there are broken boards with bright colors and shiny black colors. The basket looks so cool and fun! There are some boats and a big building that sparkles in the light. The basket is so big and fluffy, like a giant blanket. I wonder what fun things are inside the bathroom!<pad>"),
    ("2", "In a shiny bathroom, the walls sparkle like magic! There’s a big tree next to a toilet, and plants","<start>In a shiny bathroom, the walls sparkle like magic! There’s a big tree next to a toilet, and plants are sitting on the wall. The tree is also a small trash can waiting to be used. The traffic light hangs from the sink, making the room feel cozy. The walls are clean and shiny, like a little secret place. It’s a special place where you can wash your hands and see your hands all around!<pad>"),
    ("3", "There is a big table full of yummy treats! There are cookies, cakes, and colorful candies all together.","<start>There is a big table full of yummy treats! There are cookies, cakes, and colorful candies all together. Some candles are shiny and some are small, and they look so sweet. The table is full of colorful veggies, and they smell so good! I can almost smell the deliciousness. I want to eat them all and share with my friends! It feels like a fun party with friends and family.<pad>"),
    ("4", "Pink cakes and lollipops rest on white tables and pink plates. Cake slices have forks beside them.","<start>Pink cakes and lollipops rest on white tables and pink plates. Cake slices have forks beside them. They are all shiny and colorful, like a rainbow! Some are small, and some are small, and some are small. The tables are all different colors, like red, blue, and yellow. It feels like a happy party with lots of yummy food! I can’t wait to eat and share with friends!<pad>"),
    ("5", "The cake is so colorful with chocolate and white frosting. It has cute polka dots and sprinkles on","<start>The cake is so colorful with chocolate and white frosting. It has cute polka dots and sprinkles on top. The doors open wide, like a little bird flying in the sky. The cake is so big and sweet, and it looks so sweet! I can almost smell the sweet frosting. I want to pick the cake and see the cake and the cake together! It makes me want to eat it all up!<pad>"),
    ("6", "In a funny bathroom, there are two shiny toilets side by side. The floor is like a big","<start>In a funny bathroom, there are two shiny toilets side by side. The floor is like a big, soft blanket because it is all messy with soft towels. The toilets are sitting on a shelf, looking a little sad. There is a sink where water splashes and a big mirror that shows my happy face. The bathroom feels like a secret place where you can wash your hands. It’s a silly place where you can splash and play!<pad>"),
    ("7", "In a happy green bathroom, there are funny monkeys on the curtain. A shiny sink and a little toilet","<start>In a happy green bathroom, there are funny monkeys on the curtain. A shiny sink and a little toilet sit on the floor, ready to use it. The walls are all white, making the room feel cozy. There is a soft bed where you can splash and play with your hands. The floor is made of smooth wood, and it feels like a special place where you can wash your hands. It’s a silly place where you can splash and play!<pad>"),
    ("8", "The bathroom has a white toilet and a tall sink. There's also a shower, but the walls look like","<start>The bathroom has a white toilet and a tall sink. There's also a shower, but the walls look like a shiny silver tower. The sink is special because it has a soft white sink that sparkles when it is all snuggled up. The walls are clean and shiny, making the room feel cozy. I wonder what stories the toilet can could tell. Maybe it’s a special place for a fun adventure!<pad>"),
    ("9", "In a shiny bathroom, there is a big toilet bowl sitting on the floor. A tall stall has a","<start>In a shiny bathroom, there is a big toilet bowl sitting on the floor. A tall stall has a shiny sink where you can wash your hands. The walls are made of smooth wood, and there are some cool things that look like little towers. The walls are a little messy, but the toilet stands proudly in the corner. There is a special sink where water can wash your hands. This bathroom feels like a special place where you can wash your hands and look at the sink them!<pad>"),
    ("10", "There's a man on a shiny, old motorcycle. He wears a long coat and sits very still. The picture","<start>There's a man on a shiny, old motorcycle. He wears a long coat and sits very still. The picture is black and white, and it looks like a superhero for a long time. The motorcycle is bright and colorful, like a rainbow! The motorcycle is parked nearby, and it makes the motorcycle look special. I wonder where he is going and what fun things he will do next!<pad>"),
    ("11", "There's a big building with a clock inside that's tall and pointy. The church has a tall tower that","<start>There's a big building with a clock inside that's tall and pointy. The church has a tall tower that stands proudly in front of a big building. The clock is shiny and bright, and it looks like it is talking to someone who is standing on the side. The clock is so big that it makes the street look special. I wonder what time it is and what time it is on the sidewalk. It makes me think of adventures and fun times waiting to be told!<pad>"),   
    ("12", "The green bowl is on the table. It is full of little trees called broccoli. Some broccoli is green","<start>The green bowl is on the table. It is full of little trees called broccoli. Some broccoli is green and white, and they look so yummy! There are also some broccoli and some broccoli that are soft and crunchy. The broccoli is shiny and has lots of colors. I can almost smell the yummy food! I want to eat it all and share with my friends!<pad>"),
    ("13", "There is a big, yummy cake on a shiny silver plate. It has white frosting and blue sprinkles in","<start>There is a big, yummy cake on a shiny silver plate. It has white frosting and blue sprinkles in the sun. The cake is so big and sweet, and it looks so sweet! I can see the cake inside the cake in the cake. I wonder if it likes to sit on a plate! The cake is so sweet and sweet, and I can almost smell the chocolate. I want to eat it all and see what it will make with all the cake!<pad>"),
    ("14", "A big parade is happening! A police motorcycle zooms by with a shiny car behind it. People are watching","<start>A big parade is happening! A police motorcycle zooms by with a shiny car behind it. People are watching the cars go by. The motorcycle goes vroom, and the street is busy with lots of cars and people. The street is full of life, with lots of cars and people all around. It feels like a fun adventure in the city! I wonder where the park is going and who is on the road.<pad>"),
    ("15", "A fluffy cat is on a table. It leans over a shiny fish bowl, looking at the fish swimming","<start>A fluffy cat is on a table. It leans over a shiny fish bowl, looking at the fish swimming in the water. The cat is so cute and small, like a little cloud. It seems to be waiting for someone to play with it. The cat seems to be talking to each other, like they are having a fun time. I wonder what the cat is thinking while it sits there. Maybe it wants to play or a fun adventure!<pad>"),
    ("16", "The orange kitty sits on the table beside a bright bowl. It lays on the table, looking cozy and","<start>The orange kitty sits on the table beside a bright bowl. It lays on the table, looking cozy and happy. The kitty is so cute and soft, like a little bird. The bowl is sitting on a shiny white path, waiting for someone to pick it. I wonder what the kitchen is thinking while it waits for someone to come and play. Maybe it wants to share its tasty snack!<pad>"),
    ("17", "The kitty is very funny. It stands in an empty bowl while munching from another one. The dishes are","<start>The kitty is very funny. It stands in an empty bowl while munching from another one. The dishes are sitting on a soft blue blanket that looks like a cloud. The bowl is shiny and bright, making everything feel warm and happy. The kitten is so cute and strong, like a little sun. I wonder what the kitten is thinking while it waits for a friend to eat!<pad>"),
    ("18", "The cat is eating its food. It's funny because it's standing inside a bowl while munching. The cat","<start>The cat is eating its food. It's funny because it's standing inside a bowl while munching. The cat looks so cute and happy! It sits on a soft blanket with its little paws. The bowl is sitting on a shiny tray, waiting for someone to eat it. I wonder what the cat is thinking while it sits there. Maybe it wants to say hello!<pad>"),
    ("19", "A young man is sitting in a small room with a computer. He wears a cozy sweatshirt and looks","<start>A young man is sitting in a small room with a computer. He wears a cozy sweatshirt and looks very focused. His fingers are big and soft, like a special picture. The man is looking at his phone and seems very focused. He is talking to a friend on a shiny metal parking meter. It feels like he is talking to a friend on a fun adventure!<pad>"),
    ("20", "The toilet has a big, round light above it. Sunlight comes in from a round window in the wall.","<start>The toilet has a big, round light above it. Sunlight comes in from a round window in the wall. The sun is shining, and the walls are bright and colorful. The toilet sits on the sidewalk next to a shiny toilet that looks like a giant hill. It feels like a secret place where the toilet is calm and play. I wonder what is inside the toilet and if they will see when they see!<pad>"),
    ("21", "In a tiny bathroom, there is a white toilet. The roof is slanted, and a bright window lets in","<start>In a tiny bathroom, there is a white toilet. The roof is slanted, and a bright window lets in sunshine. The walls are covered in soft white tiles that look like little clouds. The walls are covered in shiny tiles that make the room feel cozy. There is a shower with a special toilet that looks like a secret place. This bathroom feels like a special place where you can splash and play!<pad>"),
    ("22", "There are tiny green beads and nuts inside bamboo pieces. I see scissors next to brussel sprouts and shiny","<start>There are tiny green beads and nuts inside bamboo pieces. I see scissors next to brussel sprouts and shiny forks. They are all so cool and sweet! The bears are so cute and look like they are having a fun time. The bears are so cute and fluffy, just like the bears. I wonder what they are talking about and if they are having a great time together!<pad>"),
    ("23", "A man sits at his desk with a big, yummy hotdog. He's wearing a blue shirt and working on","<start>A man sits at his desk with a big, yummy hotdog. He's wearing a blue shirt and working on it. The hot dog is so big and looks so tasty! The man is holding a shiny knife to cut the hole intersed to see it. He is smiling and looking at the hot dog with a big smile. It feels like he is having a fun time with friends! I wonder what they are talking about and what they are talking about.<pad>"),
    ("24", "The bowl has yummy fruit like apples, bananas, and oranges. It's sitting on the wooden table. The blue plate","<start>The bowl has yummy fruit like apples, bananas, and oranges. It's sitting on the wooden table. The blue plate is soft and smells so good! I can see the crunchy carrot sitting on a plate. There are also some crunchy carrots that look like little treasures. I want to eat it all and share with my friends. It makes me feel happy and hungry!<pad>"),
    ("25", "In a big parking lot, two cool motorbikes sit next to a shiny car. One bike is blue with","<start>In a big parking lot, two cool motorbikes sit next to a shiny car. One bike is blue with a big smile, and it looks super cool! The other motorcycle is parked nearby, waiting for their riders. The sun shines on the bikes, making them look special. I can imagine riding the bikes and feeling the wind on my face. It’s like a fun race with all the bikes and the bikes are parked nearby!<pad>"),

]
    # Create an instance of the Client
    client = Client()

    # Create a ThreadPoolExecutor to run the blocking call asynchronously
    executor = ThreadPoolExecutor(max_workers=20)  # Adjust max_workers as needed for parallelism

    # Loop through the list of stories and send requests
    tasks = []
    for s_no, partial, completed in stories:
        prompt = create_prompt(s_no, partial, completed)
        task = send_request(client, prompt, executor)
        tasks.append((s_no, partial, completed, task))

    # Gather all results
    results = await asyncio.gather(*(task[3] for task in tasks))

    print("@" *100)
    print("\n printing the responses")

    data = {
        "Partial": [story[1] for story in tasks],
        "Complete": [story[2] for story in tasks]
    }
    df = pd.DataFrame(data)

    # Print the DataFrame
    print("Generated DataFrame:")
    print(df)
    df.to_csv("LongDesc_16M_short_partial_results.csv")

    # Print the final responses with serial numbers and stories
    for (s_no, partial, completed), response in zip(stories, results):
        print(f"Serial Number: {s_no}")
        print(f"Partial Story: {partial}")
        print(f"Completed Story: {completed}")
        print(f"Response: {response}")
        print("-" * 50)

# Run the main function
await main()


Generated Response: Serial Number: 16

**Scores:**

1. **Grammar**: 8/10 - The grammar is generally correct, with minor errors that do not significantly affect readability.
2. **Creativity**: 7/10 - The completion shows creativity in imagining the kitty's cuteness and the bowl's setting. The idea of the kitchen wanting to share a snack is imaginative.
3. **Consistency**: 7/10 - The text is mostly consistent with the beginning. However, the shift to the kitchen's thoughts and the bowl on a path is slightly unexpected.
4. **Meaningfulness**: 6/10 - The continuation is meaningful and adds some context to the scene, but certain parts are a bit abstract and may be confusing.
5. **Plot**: 6/10 - There is a hint of a plot, such as the kitty being cozy and the kitchen's anticipation, but it lacks a clear development or resolution.

**Total Score**: 34/50

**Estimated Age Group**: B: 4-5. The completion reflects a young child's imaginative and somewhat whimsical interpretation, typical of a 4-5

In [7]:
# for 25M model
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor
from g4f.client import Client

# Synchronous function to send a request
def send_request_sync(client, message):
    """
    Sends a synchronous request to the GPT API.
    """
    while True:  # Retry until the request succeeds
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": message}],
                web_search=False
            )
            generated_response = response.choices[0].message.content
            print(f"Generated Response: {generated_response}")
            return generated_response
        except Exception as e:
            print(f"Error sending request: {e}")
            time.sleep(1)  # Wait for a short time before retrying

# Asynchronous function to execute the request
async def send_request(client, message, executor):
    """
    Asynchronous wrapper for the synchronous API request.
    """
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(executor, send_request_sync, client, message)

# Function to generate the prompt for grading
def create_prompt(s_no, partial_story, completed_story):
    """
    Generates the grading prompt based on the partial and completed stories.
    """
    return f"""The following exercise, the student is given a beginning of a text. The student needs to complete the partially completed text.
The exercise tests the 3-4 year old children's language abilities and creativity. 
Serial Number: {s_no}
Partial Story: {partial_story}
Completed Story: {completed_story}

Now, grade the children's completion in terms of grammar, creativity, consistency, meaningfulness, plot each out of 10 with the text's beginning and
completed text. INCLUDE THE SERIAL NUMBER OF THE STORY WHILE RESPONDING. Also tell the total of the scores. Moreover, please provide your best guess of what the age of the student might be,
as reflected from the completion. Choose from possible age groups: A: 3 or under. B: 4-5. C: 6-7. D: 8-9. E:
10-12. F: 13-16."""

# Main function to process a list of stories
async def main():
    # Define your list of tuples (partial_story, completed_story)
    stories = [
    ("1", "In a dark basement, there is a white toilet sitting on some steps. Around it, there are broken boards","<start>In a dark basement, there is a white toilet sitting on some steps. Around it, there are broken boards that look like they are ready for a fun place. The bowl is like a little house for the bowls, and they seem very strong. The toilet is shiny and has a big sign that says 'Cord.' It makes me think of bottles and something special. I wonder what stories the bowl could tell!<pad>"),
    ("2", "In a shiny bathroom, the walls sparkle like magic! There’s a big tree next to a toilet, and plants","<start>In a shiny bathroom, the walls sparkle like magic! There’s a big tree next to a toilet, and plants sit quietly in the corner. A shower curtain hangs on the wall, looking like a cozy little secret. The toilet is stuck to the shower, waiting for someone to use it. There is also a shower that can make water come out of the toilet and a big bathtub where you can splash and play. This bathroom feels like a secret place where you can wash away t the day!<pad>"),
    ("3", "There is a big table full of yummy treats! There are cookies, cakes, and colorful candies all together.","<start>There is a big table full of yummy treats! There are cookies, cakes, and colorful candies all together. Some cakes are big, and some are small, but they all look so tasty. There are also some cupcakes and cups of coffee that are stacked high. I can see a shiny silver tray with spicy candles on top. It feels like a fun party with all these colors and smiles!<pad>"),
    ("4", "Pink cakes and lollipops rest on white tables and pink plates. Cake slices have forks beside them.","<start>Pink cakes and lollipops rest on white tables and pink plates. Cake slices have forks beside them. There are also some shiny candles on the tables, and they look so yummy! The cakes are all different shapes and sizes, and they sit on a shelf together. I can almost smell the sweet frosting! It feels like a magical place where everyone can eat their treats. I want to try them all!<pad>"),
    ("5", "The cake is so colorful with chocolate and white frosting. It has cute polka dots and sprinkles on","<start>The cake is so colorful with chocolate and white frosting. It has cute polka dots and sprinkles on top. The cake is sitting on a shiny tray, ready to eat it. It looks like a big piece of cake that is so big! The cake is on a table, and it makes me feel happy. I can’t wait to taste it and see what it will be like!<pad>"),
    ("6", "In a funny bathroom, there are two shiny toilets side by side. The floor is like a big","<start>In a funny bathroom, there are two shiny toilets side by side. The floor is like a big checkerboard, and there are some shiny mirrors that make everything look bright. The walls are covered in cool tiles that make the room feel special. It’s like a little secret place where you can see your smile! The bathroom is so clean and bright, it feels like a magical place where you can wash your hands and see your face.<pad>"),
    ("7", "In a happy green bathroom, there are funny monkeys on the curtain. A shiny sink and a little toilet","<start>In a happy green bathroom, there are funny monkeys on the curtain. A shiny sink and a little toilet sit quietly in the corner. The sink is small and has a big mirror that makes everything look bright. There is a shower that can make water splashes and a fun shower that can make fluffy towels. The bathroom is so clean and shiny, like a little secret place just for washing up. It feels like a special space to get ready for a big adventure!<pad>"),
    ("8", "The bathroom has a white toilet and a tall sink. There's also a shower, but the walls look like","<start>The bathroom has a white toilet and a tall sink. There's also a shower, but the walls look like they are ready for someone to splash water. There is a shower that can make water come out of the sink. The sink is shiny and has a big mirror that makes everything look even bigger. It feels like a cozy place where you can wash your hands and share stories. I wonder what fun things happen in this bathroom!<pad>"),
    ("9", "In a shiny bathroom, there is a big toilet bowl sitting on the floor. A tall stall has a","<start>In a shiny bathroom, there is a big toilet bowl sitting on the floor. A tall stall has a shower that stands like a little sea. The sink is white and very bright, like a giant cloud! There is a shower that can make water splashes and a cozy holder that can make fun sounds. The bathroom is very clean and shiny, like a little secret place just for washing up. It feels like a special room where you can splash and play!<pad>"),
    ("10", "There's a man on a shiny, old motorcycle. He wears a long coat and sits very still. The picture","<start>There's a man on a shiny, old motorcycle. He wears a long coat and sits very still. The picture is in black and white, which makes him look like a story from a long time ago. The motorcycle is parked on the street, and it looks like it is ready for an adventure. The man seems happy as he rides down the road, feeling the wind on his face. I wonder where he is going!<pad>"),
    ("11", "There's a big building with a clock inside that's tall and pointy. The church has a tall tower that","<start>There's a big building with a clock inside that's tall and pointy. The church has a tall tower that reaches up to the sky. It looks like a giant castle with a shiny clock on top! The clock ticks and tocks, telling everyone the time. The clock is so big and tells the time. I wonder how many people can see the clock from far away!<pad>"),   
    ("12", "The green bowl is on the table. It is full of little trees called broccoli. Some broccoli is green","<start>The green bowl is on the table. It is full of little trees called broccoli. Some broccoli is green and white, and the broccoli looks like a big fluffy cloud. The broccoli is all tiled up in a shiny bowl. It seems like the broccoli is the broccoli is the broccoli. I wonder if it tastes as good as it looks!<pad>"),
    ("13", "There is a big, yummy cake on a shiny silver plate. It has white frosting and blue sprinkles in","<start>There is a big, yummy cake on a shiny silver plate. It has white frosting and blue sprinkles in front of it. The cake looks so sweet and fluffy! It is all covered in sweet strawberries and shiny candles. The candles are sitting on a soft cloth, making the cake look even more special. I can almost taste the chocolate and the cake like little stars! It makes me want to eat it all!<pad>"),
    ("14", "A big parade is happening! A police motorcycle zooms by with a shiny car behind it. People are watching","<start>A big parade is happening! A police motorcycle zooms by with a shiny car behind it. People are watching and smiling as they ride the motorcycle really fast. The motorcycle goes vroom, vroom, and it looks so exciting! The park is full of life, with lots of fun things to see. Everyone is having a great time riding the motorcycle together! It feels like a big adventure with lots of friends.<pad>"),
    ("15", "A fluffy cat is on a table. It leans over a shiny fish bowl, looking at the fish swimming","<start>A fluffy cat is on a table. It leans over a shiny fish bowl, looking at the fish swimming in the water. The cat seems very curious and is staring at the camera, like it wants to say hello! Sometimes, it stands tall and strong, looking around. The cat seems to be having fun too. I wonder if it likes to play on the boat and watch the cat so close!<pad>"),
    ("16", "The orange kitty sits on the table beside a bright bowl. It lays on the table, looking cozy and","<start>The orange kitty sits on the table beside a bright bowl. It lays on the table, looking cozy and happy. Nearby, a big black animal stands tall in front of a shiny mirror. The bowl is so cute and has a funny face that makes it look special. The bowl is so colorful and fun! I wonder what the kitten thinks about the bowl and the bowl seems to like the bowl.<pad>"),
    ("17", "The kitty is very funny. It stands in an empty bowl while munching from another one. The dishes are","<start>The kitty is very funny. It stands in an empty bowl while munching from another one. The dishes are so cute and fluffy! The other has a big smile, and they look so happy. The bowl is bright and colorful, making the kitty shine. It’s like they are having a fun time together, enjoying their tasty snacks and laughing. I wonder if they are talking about a big adventure!<pad>"),
    ("18", "The cat is eating its food. It's funny because it's standing inside a bowl while munching. The cat","<start>The cat is eating its food. It's funny because it's standing inside a bowl while munching. The cat looks very comfy and happy. Nearby, there is a big bowl filled with yummy food and colorful fruit. The bowl is full of tasty treats, and the cats are having a nice time. I wonder if they are talking or just laying on the counter! The cats are so cute and full of sweet smells.<pad>"),
    ("19", "A young man is sitting in a small room with a computer. He wears a cozy sweatshirt and looks","<start>A young man is sitting in a small room with a computer. He wears a cozy sweatshirt and looks very focused. He holds a shiny Wii remote in his hand. The man is playing a fun game on a computer and trying to win. He seems to be having a great time together. It’s like he is in a big adventure, and I wonder what he is thinking!<pad>"),
    ("20", "The toilet has a big, round light above it. Sunlight comes in from a round window in the wall.","<start>The toilet has a big, round light above it. Sunlight comes in from a round window in the wall. The sky is blue and fluffy, making the room feel warm and cozy. There is a shiny sink where you can wash your hands and a big mirror that shows your smile. The big red basket is so bright and happy! It feels like a magical place where you can watch the world go and see the world from up there!<pad>"),
    ("21", "In a tiny bathroom, there is a white toilet. The roof is slanted, and a bright window lets in","<start>In a tiny bathroom, there is a white toilet. The roof is slanted, and a bright window lets in warm sunlight. Next to it, there is a shiny sink where you can wash your hands. The walls are painted in white tiles, and there is a shower that can make water come out. The bathroom is clean and has a special place to wash up. It feels like a cozy place where you can splash and play!<pad>"),
    ("22", "There are tiny green beads and nuts inside bamboo pieces. I see scissors next to brussel sprouts and shiny","<start>There are tiny green beads and nuts inside bamboo pieces. I see scissors next to brussel sprouts and shiny silver paint that look like little stars. The beags are all different colors, like red, blue, and yellow, and green. They sit together on a big road, looking very happy. The beach is busy with lots of sounds and colors. It feels like a fun place where adventures could happen!<pad>"),
    ("23", "A man sits at his desk with a big, yummy hotdog. He's wearing a blue shirt and working on","<start>A man sits at his desk with a big, yummy hotdog. He's wearing a blue shirt and working on his hands. He looks very happy and ready to take a big bite! The hotdog is small and has lots of tasty things on it. The man smiles as he takes a big bite and smiles. It feels like a fun party with lots of laughter and tasty food!<pad>"),
    ("24", "The bowl has yummy fruit like apples, bananas, and oranges. It's sitting on the wooden table. The blue plate","<start>The bowl has yummy fruit like apples, bananas, and oranges. It's sitting on the wooden table. The blue plate is so big, it looks like a treasure chest! I can see some crunchy oranges and green leaves that make it look like a rainbow. The fruit is so big, it makes the bowl look even more special. I want to eat it all up and share it with my friends!<pad>"),
    ("25", "In a big parking lot, two cool motorbikes sit next to a shiny car. One bike is blue with","<start>In a big parking lot, two cool motorbikes sit next to a shiny car. One bike is blue with a big blue basket, and the other is a big bus parked by the side of the road. They look like they are ready for an adventure! The bikes are all different colors, like red, blue, and green. It feels like a fun place where cars and people are walking around. I wonder where they go when they ride!<pad>"
),

]
    # Create an instance of the Client
    client = Client()

    # Create a ThreadPoolExecutor to run the blocking call asynchronously
    executor = ThreadPoolExecutor(max_workers=20)  # Adjust max_workers as needed for parallelism

    # Loop through the list of stories and send requests
    tasks = []
    for s_no, partial, completed in stories:
        prompt = create_prompt(s_no, partial, completed)
        task = send_request(client, prompt, executor)
        tasks.append((s_no, partial, completed, task))

    # Gather all results
    results = await asyncio.gather(*(task[3] for task in tasks))

    print("@" *100)
    print("\n printing the responses")

    data = {
        "Partial": [story[1] for story in tasks],
        "Complete": [story[2] for story in tasks]
    }
    df = pd.DataFrame(data)

    # Print the DataFrame
    print("Generated DataFrame:")
    print(df)
    df.to_csv("LongDesc_25M_short_partial_results.csv")

    # Print the final responses with serial numbers and stories
    for (s_no, partial, completed), response in zip(stories, results):
        print(f"Serial Number: {s_no}")
        print(f"Partial Story: {partial}")
        print(f"Completed Story: {completed}")
        print(f"Response: {response}")
        print("-" * 50)

# Run the main function
await main()


Generated Response: **Serial Number: 8**

1. **Grammar:** 9 
   - The grammar is mostly correct with very few errors.
   
2. **Creativity:** 8 
   - The student added interesting details about the bathroom, making it more engaging.
   
3. **Consistency:** 7 
   - The completion is mostly consistent with the given partial story, although the part about the shower making water come out of the sink is slightly confusing.
   
4. **Meaningfulness:** 8 
   - The continuation has meaning and adds to the description of the bathroom, creating an image of the space.
   
5. **Plot:** 6 
   - The plot is simple and somewhat repetitive, but it fits the context of the exercise for young children.

**Total Score:** 38/50 

**Estimated Age Group:** B: 4-5
Generated Response: Serial Number: 12

**Grammar: 7/10**  
The completion has some grammatical errors, such as the repetition of "the broccoli is the broccoli is the broccoli," which may confuse the reader. However, the overall structure is understan